# Concatenação de features offline por dataset

Este notebook concatena as features offline dos modelos **classical** e **quantum** para cada dataset:
- BreastMNIST
- BUSUCLM
- INbreast

Para cada split (`train`, `val`, `test`), ele:
1. carrega as features de cada branch;
2. valida consistência de labels;
3. concatena no eixo de features (`axis=1`);
4. salva em `experiments/models/fusion/offline/<dataset>/concatenated/`.

In [5]:
from pathlib import Path
import numpy as np

SPLITS = ("train", "val", "test")
DATASETS = ("BreastMNIST", "BUSUCLM", "INbreast")

BRANCHES = {
    "classical_resnet": ("classical", "resnet"),
    "classical_shallow": ("classical", "shallow"),
    "quantum_basic": ("quantum", "basic"),
    "quantum_trainable": ("quantum", "trainable"),
}

# Combinações par a par: quantum + classical
COMBINATIONS = [
    ("quantum_basic", "classical_shallow", "basic_shallow"),
    ("quantum_basic", "classical_resnet", "basic_resnet"),
    ("quantum_trainable", "classical_shallow", "trainable_shallow"),
    ("quantum_trainable", "classical_resnet", "trainable_resnet"),
]

OFFLINE_ROOT = Path("/home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/models/fusion/offline")
print(f"Offline root: {OFFLINE_ROOT}")
print(f"\nCombinações a serem processadas:")
for q, c, name in COMBINATIONS:
    print(f"  - {name}: {q} + {c}")

Offline root: /home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/models/fusion/offline

Combinações a serem processadas:
  - basic_shallow: quantum_basic + classical_shallow
  - basic_resnet: quantum_basic + classical_resnet
  - trainable_shallow: quantum_trainable + classical_shallow
  - trainable_resnet: quantum_trainable + classical_resnet


In [6]:
# Carregar labels de referência da pasta centralizada
LABELS_ROOT = Path("/home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/labels")
reference_labels = {}

print(f"Carregando labels de referência de: {LABELS_ROOT}\n")

for dataset_name in DATASETS:
    reference_labels[dataset_name] = {}
    for split in SPLITS:
        y_path = LABELS_ROOT / dataset_name / f"{split}_labels.npy"
        if y_path.exists():
            y = np.load(y_path)
            reference_labels[dataset_name][split] = y
            print(f"{dataset_name} | {split:5s} - shape: {y.shape} - unique: {np.unique(y)}")
        else:
            print(f"{dataset_name} | {split:5s} - ARQUIVO NÃO ENCONTRADO: {y_path}")

print(f"\nLabels carregados com sucesso!")
print(f"Acesso: reference_labels[dataset][split]")

Carregando labels de referência de: /home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/labels

BreastMNIST | train - shape: (546, 1) - unique: [0 1]
BreastMNIST | val   - shape: (78, 1) - unique: [0 1]
BreastMNIST | test  - shape: (156, 1) - unique: [0 1]
BUSUCLM | train - shape: (546, 1) - unique: [0 1]
BUSUCLM | val   - shape: (78, 1) - unique: [0 1]
BUSUCLM | test  - shape: (156, 1) - unique: [0 1]
INbreast | train - shape: (546, 1) - unique: [0 1]
INbreast | val   - shape: (78, 1) - unique: [0 1]
INbreast | test  - shape: (156, 1) - unique: [0 1]

Labels carregados com sucesso!
Acesso: reference_labels[dataset][split]


In [7]:
def load_branch_features(dataset: str, branch_key: str, split: str):
    """Carrega apenas as features de um branch específico"""
    family, model = BRANCHES[branch_key]
    folder = OFFLINE_ROOT / dataset / family / model

    x_path = folder / f"{split}_features.npy"

    if not x_path.exists():
        raise FileNotFoundError(f"Feature file não encontrado: {x_path}")

    x = np.load(x_path)
    return x


def concat_pair_and_save(dataset: str, quantum_key: str, classical_key: str, combination_name: str):
    """Concatena um par quantum + classical e salva"""
    output_dir = OFFLINE_ROOT / dataset / "concatenated" / combination_name
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"  {combination_name}:")

    for split in SPLITS:
        # Carrega features de cada branch
        x_quantum = load_branch_features(dataset, quantum_key, split)
        x_classical = load_branch_features(dataset, classical_key, split)

        # Concatena features no eixo 1 (features)
        x_concat = np.concatenate([x_quantum, x_classical], axis=1)
        
        # Usa os labels de referência (centralizado)
        y_ref = reference_labels[dataset][split]

        # Valida consistência
        if x_concat.shape[0] != y_ref.shape[0]:
            raise ValueError(
                f"Inconsistência entre features concatenadas e labels em {dataset} | {split}: "
                f"features={x_concat.shape[0]} labels={y_ref.shape[0]}"
            )

        x_out = output_dir / f"{split}_features_concat.npy"
        y_out = output_dir / f"{split}_labels.npy"

        np.save(x_out, x_concat)
        np.save(y_out, y_ref)

        print(
            f"    [{split}] concat shape: {x_concat.shape} | "
            f"quantum: {x_quantum.shape[1]} + classical: {x_classical.shape[1]} = {x_concat.shape[1]}"
        )


def process_all_combinations_for_dataset(dataset: str):
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset}")
    print(f"{'='*60}")

    for quantum_key, classical_key, combination_name in COMBINATIONS:
        concat_pair_and_save(dataset, quantum_key, classical_key, combination_name)
    
    print(f"\n✓ {dataset} concluído!")

In [8]:
for dataset_name in DATASETS:
    process_all_combinations_for_dataset(dataset_name)

print("\n" + "="*60)
print("✓ CONCLUÍDO: Features offline concatenadas par a par!")
print("="*60)
print("\nPastas criadas:")
for dataset_name in DATASETS:
    for _, _, comb_name in COMBINATIONS:
        path = OFFLINE_ROOT / dataset_name / "concatenated" / comb_name
        print(f"  {dataset_name}/{comb_name}/")


Dataset: BreastMNIST
  basic_shallow:
    [train] concat shape: (546, 256) | quantum: 128 + classical: 128 = 256
    [val] concat shape: (78, 256) | quantum: 128 + classical: 128 = 256
    [test] concat shape: (156, 256) | quantum: 128 + classical: 128 = 256
  basic_resnet:
    [train] concat shape: (546, 256) | quantum: 128 + classical: 128 = 256
    [val] concat shape: (78, 256) | quantum: 128 + classical: 128 = 256
    [test] concat shape: (156, 256) | quantum: 128 + classical: 128 = 256
  trainable_shallow:
    [train] concat shape: (546, 256) | quantum: 128 + classical: 128 = 256
    [val] concat shape: (78, 256) | quantum: 128 + classical: 128 = 256
    [test] concat shape: (156, 256) | quantum: 128 + classical: 128 = 256
  trainable_resnet:
    [train] concat shape: (546, 256) | quantum: 128 + classical: 128 = 256
    [val] concat shape: (78, 256) | quantum: 128 + classical: 128 = 256
    [test] concat shape: (156, 256) | quantum: 128 + classical: 128 = 256

✓ BreastMNIST concl